# Fire Loss Analysis By Month

## Hypothesis

Due to colder temperatures and people spending more time in houses (especially vacant properties), there will be more fires with property loss and more monetary damages done during colder months than during warmer months. 

## Data Sources

### Fire Property Damage Data

Source: https://data.louisvilleky.gov/dataset/fire-property-damage

### Weather Data

Source: http://www.intellicast.com/Local/History.aspx?location=USKY1096


In [1]:
import datetime
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# *Requirement* SQL database
# Create the database using relative path
con = sqlite3.connect('FireLoss.db3')

# *Requirement* Python script to fetch data and load it into database
# Create table - properly handle CSV headers and skip the header row issue
df = pd.read_csv('PropLoss.csv', header=0, names=['ID', 'Date', 'Total_Loss'], skiprows=1, parse_dates=['Date'])
df.to_sql("propertyloss", con, if_exists='replace', index=False)
cur = con.cursor()
con.commit()

In [ ]:
# *Requirement* Script to retrieve data from database
# Query & group data to get Total Property Loss in $ by Month of the Year
# Filter out null/zero losses for cleaner analysis

results = cur.execute("""SELECT strftime('%m', Date) AS month, SUM(Total_Loss/1000.0) as sum_prop_loss_thousands 
    FROM propertyloss 
    WHERE Total_Loss > 0 
    GROUP BY month
    ORDER BY month""")

In [ ]:
# *Requirement* Python script to modify data to prepare for visualization
# Add Total Property Loss data to lists to prepare for visualization

months = []
sum_prop_loss_by_month = []
for row in results:
    sum_prop_loss_by_month.append(row[1])
    months.append(row[0])

In [ ]:
# *Requirement* Visualize analysis
# Visualize Total Property Loss in $ aggregated by month of the year 

plt.figure(figsize=(10, 6))
plt.bar(months, sum_prop_loss_by_month, color='red')
plt.ylabel("Property Loss in $ / 1000")
plt.xlabel("Month")
plt.title("Total Property Loss by Month of the Year")

# Save figure BEFORE showing it (plt.show() clears the figure)
plt.savefig('FireLossByMonthofYear.png', dpi=300, bbox_inches='tight')
plt.show()

## Average Temperatures in Louisville by Month

| Month | Avg Low | Avg High | 
|-------|---------|----------| 
| 1     | 25°     | 41°      | 
| 2     | 29°     | 47°      | 
| 3     | 37°     | 57°      | 
| 4     | 46°     | 67°      | 
| 5     | 56°     | 75°      | 
| 6     | 65°     | 83°      | 
| 7     | 70°     | 87°      | 
| 8     | 68°     | 86°      | 
| 9     | 61°     | 79°      | 
| 10    | 49°     | 68°      | 
| 11    | 39°     | 56°      | 
| 12    | 30°     | 45°      | 

## Analysis

* It appears that my original hypothesis was generally correct. Based on an initial analysis of the data, the colder months (months with average lows 46 degrees Farenheit or lower) have more fire damage, with the exception of September.

* If I were to continue this project, I would want to do an analysis of the causes of fires during the colder months to see if there was a root cause of the higher rates of property damage. My initial hypothesis is there are more fires starting in vacant properties due to more fires started by squatters to keep warm. I would also want to see if there was a reason that September had a spike or if that was just a statistical anamoly. 

In [6]:
# Close the Connection
con.close()